<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.3-rag-engine/notebooks/GCP_Capstone_4.3_RAG_Engine.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.3 Vertex AI RAG Engine — Managed RAG Pipeline
**Netsetos GenAI Engineering — GCP Capstone**

Create corpus, import from GCS/Drive/Slack/Jira, retrieve, and generate with automatic citations.


## Setup


In [ ]:
!pip install -q google-cloud-aiplatform google-genai
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from vertexai import rag
import vertexai
from google import genai
from google.genai import types

vertexai.init(project=PROJECT_ID, location='us-central1')
client = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')


## Cell 1: Create a RAG Corpus


In [ ]:
embedding_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model='publishers/google/models/text-embedding-005'
    )
)

corpus = rag.create_corpus(
    display_name='documind-lesson43',
    description='Lesson 4.3 test corpus',
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_config),
)
print(f'Corpus: {corpus.name}')

# List corpora
for c in rag.list_corpora():
    print(f'  {c.display_name}: {c.name}')


## Cell 2: Import from GCS


In [ ]:
# Upload test files to GCS first:
# gsutil cp test.pdf gs://your-bucket/docs/

response = rag.import_files(
    corpus.name,
    ['gs://YOUR-BUCKET/docs/'],  # CHANGE
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512, chunk_overlap=100)),
    max_embedding_requests_per_min=900,
)
print(f'Imported: {response.imported_rag_files_count}')
print(f'Skipped: {response.skipped_rag_files_count}')


## Cell 3: Import from Google Drive


In [ ]:
# Share your Drive folder with the RAG service account first
# (Check IAM for the service account email)

# drive_response = rag.import_files(
#     corpus.name,
#     ['https://drive.google.com/drive/folders/YOUR_FOLDER_ID'],
#     transformation_config=rag.TransformationConfig(
#         chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)),
# )
# print(f'Drive import: {drive_response.imported_rag_files_count}')


## Cell 4: Direct Retrieval


In [ ]:
response = rag.retrieval_query(
    rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
    text='What is RAG?',
    rag_retrieval_config=rag.RagRetrievalConfig(
        top_k=5,
        filter=rag.Filter(vector_distance_threshold=0.5)),
)

for ctx in response.contexts.contexts:
    print(f'Source: {ctx.source_uri}')
    print(f'Score: {ctx.score:.3f}, Distance: {ctx.distance:.3f}')
    print(f'Text: {ctx.text[:150]}...\n')


## Cell 5: Grounded Generation


In [ ]:
rag_retrieval_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[types.VertexRagStoreRagResource(rag_corpus=corpus.name)],
            rag_retrieval_config=types.RagRetrievalConfig(
                top_k=5,
                filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Summarize the main topics in my documents',
    config=types.GenerateContentConfig(tools=[rag_retrieval_tool]))
print(response.text)

# Grounding metadata — automatic citations
for candidate in response.candidates:
    gm = candidate.grounding_metadata
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.retrieved_context.uri}')
    if gm and gm.grounding_supports:
        for support in gm.grounding_supports:
            print(f'  Claim: {support.segment.text}')
            print(f'  Backed by: {support.grounding_chunk_indices}')


## Cell 6: ManagedRAG Module


In [ ]:
class ManagedRAG:
    def __init__(self, project, location='us-central1'):
        vertexai.init(project=project, location=location)
        self.client = genai.Client(enterprise=True, project=project, location=location)
        self.corpus = None

    def create_corpus(self, name, description=''):
        emb = rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model='publishers/google/models/text-embedding-005'))
        self.corpus = rag.create_corpus(
            display_name=name, description=description,
            backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=emb))
        return self.corpus.name

    def use_corpus(self, corpus_name):
        self.corpus = rag.get_corpus(name=corpus_name)

    def ingest(self, paths, chunk_size=512, chunk_overlap=100):
        return rag.import_files(
            self.corpus.name, paths,
            transformation_config=rag.TransformationConfig(
                rag.ChunkingConfig(chunk_size=chunk_size, chunk_overlap=chunk_overlap)),
            max_embedding_requests_per_min=900)

    def retrieve(self, query, top_k=5):
        return rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=self.corpus.name)],
            text=query,
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=top_k, filter=rag.Filter(vector_distance_threshold=0.5)))

    def ask(self, question, model_name='gemini-3.6-flash'):
        rag_tool = types.Tool(retrieval=types.Retrieval(
            vertex_rag_store=types.VertexRagStore(
                rag_resources=[types.VertexRagStoreRagResource(rag_corpus=self.corpus.name)],
                rag_retrieval_config=types.RagRetrievalConfig(
                    top_k=5, filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))
        return self.client.models.generate_content(
            model=model_name, contents=question,
            config=types.GenerateContentConfig(tools=[rag_tool]))

print('ManagedRAG class ready')


## Cell 7: Cleanup


In [ ]:
# Delete corpus when done (saves storage costs)
# rag.delete_corpus(name=corpus.name)
# print('Corpus deleted')


## ✅ Lesson 4.3 Complete!

- ✅ Created RAG corpus with text-embedding-005
- ✅ Imported from GCS (and optionally Drive)
- ✅ Automatic chunking + embedding + indexing
- ✅ Direct retrieval with retrieval_query()
- ✅ Grounded generation with automatic citations
- ✅ ManagedRAG production module

**Module 4 Complete! Three RAG approaches built:**
- 4.1: Document AI ingestion (OCR/Layout/Form)
- 4.2: DIY RAG pipeline (manual embed→retrieve→generate)
- 4.3: Managed RAG Engine (corpus→import→query)

**Next: Module 5 — BigQuery ML & SQL-Native AI**
